# 실험 과정
            ┌────────────┐       ┌───────────────┐
Image ───►  │CNN Backbone│──┐    │ Keypoint CSV  │
            └────────────┘  │    └───────┬───────┘
                            ▼            ▼
                     [Image Feature]  [CSV Feature]
                            └────┬─────┘
                                 ▼
                          🔗 Concat Layer
                                 ▼
                        🔽 MLP Classification Head
                                 ▼
                        🏷️ Binary Classification (Good/Bad)
### 라이브러리 임포트

In [2]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.metrics import f1_score
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn as nn
import timm  # pip install timm
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import pandas as pd
from PIL import Image
import torch.utils
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class PosutreDataset(torch.utils.data.Dataset):
    def __init__(self,dataframe,image_dir,transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(image_path).convert('RGB')
        label = row['class_id']
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
# 1. 필요한 컬럼 모두 불러오기
train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")[[
    "filename", "class_id", "x_center", "y_center", "width", "height"
]]
valid_df = pd.read_csv("../dataset-modification/valid_pose_parsed.csv")[[
    "filename", "class_id", "x_center", "y_center", "width", "height"
]]

# 2. 중복된 filename에 대해 대표값 선택 (class_id는 min, bbox는 first)
train_df_grouped = train_df.groupby("filename").agg({
    "class_id": "min",       # 또는 mode, if needed
    "x_center": "first",
    "y_center": "first",
    "width": "first",
    "height": "first"
}).reset_index()

valid_df_grouped = valid_df.groupby("filename").agg({
    "class_id": "min",
    "x_center": "first",
    "y_center": "first",
    "width": "first",
    "height": "first"
}).reset_index()

train_df = train_df_grouped
valid_df = valid_df_grouped

train_image_dir = "../dataset-modification/train-visualized/images/"
valid_image_dir = "../dataset-modification/valid-visualized/images/"

# 3. 클래스 가중치 계산
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=[0, 1],
    y=train_df["class_id"]
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# 4. 확인
print("Train 클래스 분포:\n", train_df["class_id"].value_counts())
print("Class weights:", class_weights_tensor)
print("train_df columns:", train_df.columns.tolist())


Train 클래스 분포:
 0    1586
1     995
Name: class_id, dtype: int64
Class weights: tensor([0.8137, 1.2970])
train_df columns: ['filename', 'class_id', 'x_center', 'y_center', 'width', 'height']


Keypoint 전용 Dataset: KeypointPostureDataset

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset

class KeypointPostureDataset(Dataset):
    def __init__(self, csv_path, normalize=True):
        """
        :param csv_path: keypoint 포함된 CSV 파일 경로
        :param normalize: 정규화 여부 (Z-score 방식)
        """
        self.df = pd.read_csv(csv_path)
        self.normalize = normalize

        # keypoint 컬럼 자동 수집 (x, y만 사용)
        self.kpt_columns = [col for col in self.df.columns if "_x" in col or "_y" in col]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # keypoints: torch.FloatTensor (shape: [34])
        keypoints = row[self.kpt_columns].values.astype('float32')
        keypoints = torch.tensor(keypoints)

        if self.normalize:
            keypoints = (keypoints - keypoints.mean()) / (keypoints.std() + 1e-6)

        # label: torch.LongTensor (0 or 1)
        label = torch.tensor(row["class_id"], dtype=torch.long)

        return keypoints, label

class EarlyStopping:
    def __init__(self, patience=30, delta=0.0, checkpoint_path='checkpoint.pt'):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.checkpoint_path = checkpoint_path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"🟡 EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                print("🛑 EarlyStopping triggered! Stopping training.")
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        """Validation loss가 개선될 때만 모델 저장"""
        torch.save(model.state_dict(), self.checkpoint_path)
        print(f"✅ Model saved to {self.checkpoint_path}")


In [11]:
train_kpt_dataset = KeypointPostureDataset(
    csv_path="../dataset-modification/train_pose_parsed.csv",
    normalize=True
)

train_loader = torch.utils.data.DataLoader(train_kpt_dataset, batch_size=32, shuffle=True)

# 확인
for kpt, lbl in train_loader:
    print(kpt.shape)  # e.g. (32, 34)
    print(lbl.shape)  # e.g. (32,)
    break


torch.Size([32, 34])
torch.Size([32])


1. 이미지 특징 추출 모듈

In [ ]:
import torchvision.models as models

class ImageFeatureExtractor(nn.Module):
    def __init__(self, backbone_name='resnet50', out_dim=512):
        super().__init__()
        if backbone_name == 'resnet50':
            model = models.resnet50(weights='IMAGENET1K_V1')
            modules = list(model.children())[:-1]  # remove FC
            self.backbone = nn.Sequential(*modules)
            self.out_dim = model.fc.in_features  # usually 2048
        # 추가 backbone 지원 가능
        
        self.project = nn.Linear(self.out_dim, out_dim)  # optional compression

    def forward(self, x):  # x: (B, 3, 224, 224)
        x = self.backbone(x).squeeze()
        x = self.project(x)
        return x  # (B, out_dim)


2. Keypoint Encoder

In [ ]:
class KeypointEncoder(nn.Module):
    def __init__(self, input_dim=34, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, out_dim),
            nn.ReLU()
        )

    def forward(self, x):  # x: (B, 34)
        return self.net(x)


 3. Multimodal Classifier (Fusion + Head)

In [ ]:
class MultiModalClassifier(nn.Module):
    def __init__(self, img_feat_dim=512, kp_feat_dim=128, hidden_dim=128):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(img_feat_dim + kp_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2)  # binary
        )

    def forward(self, img_feat, kp_feat):
        x = torch.cat([img_feat, kp_feat], dim=1)
        return self.fusion(x)


In [ ]:
img_encoder = ImageFeatureExtractor(backbone_name='resnet50', out_dim=512)
kp_encoder = KeypointEncoder(input_dim=34, out_dim=128)
classifier = MultiModalClassifier(img_feat_dim=512, kp_feat_dim=128, hidden_dim=128)

# forward pass 예시
img_input = torch.randn(16, 3, 224, 224)     # 이미지 입력
kp_input = torch.randn(16, 34)               # keypoint (x, y)*17

img_feat = img_encoder(img_input)           # (16, 512)
kp_feat = kp_encoder(kp_input)              # (16, 128)
logits = classifier(img_feat, kp_feat)      # (16, 2)
